# 🚗 Car Market Segmentation — Unsupervised Clustering

## 📌 Project Overview

Car manufacturers, dealerships, and analysts often want to understand 
how vehicles naturally group together based on their specs and pricing — 
without predefined categories like "economy" or "luxury." Rather than 
relying on manual labels, we can let the data itself reveal these 
groupings.

In this project, we apply **K-Means clustering**, an unsupervised 
learning technique, to a dataset of ~12,000 U.S. car models 
(1990–2018), using features like price (MSRP), horsepower, fuel 
efficiency, and engine specs to uncover natural market segments.

---

## 🎯 Objective

Group cars into distinct segments based on their specifications and 
price, **without using any predefined labels**, then interpret what 
characterizes each resulting group (e.g., "high-power luxury," 
"budget-efficient compacts," "family SUVs").

---

## 🔑 Key Difference From Supervised Learning

Unlike our previous regression and classification projects, this 
project has **no target variable**. There's nothing to predict and no 
"correct answer" to check against. Instead of a train/test split and 
accuracy scores, we:

- Skip the train/test split entirely — clustering uses the full 
  dataset, since there's no prediction to validate against unseen data
- Judge success through **cluster separation and interpretability**, 
  using tools like the Elbow Method and Silhouette Score, rather than 
  accuracy or F1
- Focus on **describing and understanding** the groups the algorithm 
  finds, rather than predicting a known outcome

---

## 🗺️ Project Roadmap

1. **Data Collection** — car specs and pricing data via the Kaggle API
2. **Data Cleaning** — handle missing values, duplicates, and 
   inconsistent entries
3. **Exploratory Data Analysis** — understand distributions of price, 
   horsepower, MPG, and other specs
4. **Feature Scaling** — critical for K-Means, since it's a 
   distance-based algorithm sensitive to feature magnitude
5. **Finding Optimal K** — Elbow Method and Silhouette Score to decide 
   how many clusters best fit the data
6. **Fitting K-Means** — assign each car to a cluster
7. **Visualizing Clusters** — scatter plots and dimensionality 
   reduction (PCA) to see the groupings
8. **Profiling Clusters** — describe what makes each segment distinct 
   (e.g., average price, power, size, common vehicle styles)

---

## 📊 Data Source

This project uses the **Car Features and MSRP** dataset from Kaggle, 
containing car listings from 1990–2018, including make, model, engine 
specs, MPG, vehicle category, and MSRP.

> Source: [Car Features and MSRP — Kaggle](https://www.kaggle.com/datasets/CooperUnion/cardataset)

### 📥 Loading the Dataset and Libraries

Before we start, we need to install the necessary Python libraries and **load the dataset**.


In [1]:
%pip install pandas numpy matplotlib seaborn scikit-learn jupyterlab nbconvert kagglehub sweetviz joblib yellowbrick setuptools watermark -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ==========================================
# 1. DATA MANIPULATION & VISUALIZATION
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sweetviz as sv
import os
import kagglehub
import logging
import warnings
import kagglehub
from sweetviz import FeatureConfig

warnings.filterwarnings('ignore', category=UserWarning)
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

# ==========================================
# 2. PREPROCESSING & PIPELINES
# ==========================================
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# ==========================================
# 3. CLUSTERING & EVALUATION
# ==========================================
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
from yellowbrick.cluster import KElbowVisualizer

c:\Users\LarTI\OneDrive\Desktop\Projects\car_clustering_kmeans\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load the watermark extension to log the environment state
%reload_ext watermark

# Display professional metadata tracking our data engineering stack (NO SPACES after commas)
%watermark -a "Maykon - Car Market Segmentation 🚗" -d -u -v -p pandas,numpy,matplotlib,seaborn,scikit-learn,kagglehub,jupyterlab,nbconvert,sweetviz,joblib,yellowbrick,watermark,setuptools

Author: Maykon - Car Market Segmentation 🚗

Last updated: 2026-08-14

Python implementation: CPython
Python version       : 3.13.7
IPython version      : 9.16.1

pandas      : 3.0.5
numpy       : 2.5.2
matplotlib  : 3.11.1
seaborn     : 0.13.2
scikit-learn: 1.9.0
kagglehub   : 1.0.2
jupyterlab  : 4.6.3
nbconvert   : 7.17.1
sweetviz    : 2.3.3
joblib      : 1.5.3
yellowbrick : 1.5
watermark   : 2.6.0
setuptools  : 84.0.0



In [4]:
# --- AUTOMATED KAGGLE INGESTION ---
# Download the latest version of the specific student behavioral dataset
path = kagglehub.dataset_download("CooperUnion/cardataset")
print("🚀 Path to dataset files:", path)

🚀 Path to dataset files: C:\Users\LarTI\.cache\kagglehub\datasets\CooperUnion\cardataset\versions\1


In [5]:
# --- LOCATING AND READING THE CSV ---
# List out all files inside the downloaded repository path to spot the target file
all_files = os.listdir(path)
print("📂 Files discovered in directory:", all_files)

# Filter out all CSV files dynamically
csv_files = [file for file in all_files if file.endswith('.csv')]

if len(csv_files) == 0:
    raise FileNotFoundError("❌ Critical Error: No CSV files found in the downloaded folder!")
else:
    # Grab the primary CSV file found
    csv_filename = csv_files[0]
    full_csv_path = os.path.join(path, csv_filename)
    print(f"🎯 Target CSV located: {csv_filename}")

📂 Files discovered in directory: ['data.csv']
🎯 Target CSV located: data.csv


In [6]:
# Ingest the dataset into a pandas DataFrame
df = pd.read_csv(full_csv_path)
print(f"✅ Dataset successfully loaded! Named 'df', Shape: {df.shape[0]} rows, {df.shape[1]} columns.")

✅ Dataset successfully loaded! Named 'df', Shape: 11914 rows, 16 columns.


In [7]:
df.head(10)

,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Market Category,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
0,BMW,1 Series M,2011,premium unleaded (required),335.0,6.0,MANUAL,rear wheel drive,2.0,"Factory Tuner,Luxury,High-Performance",Compact,Coupe,26,19,3916,46135
1,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Convertible,28,19,3916,40650
2,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,High-Performance",Compact,Coupe,28,20,3916,36350
3,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Coupe,28,18,3916,29450
4,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,Luxury,Compact,Convertible,28,18,3916,34500
5,BMW,1 Series,2012,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Coupe,28,18,3916,31200
6,BMW,1 Series,2012,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Convertible,26,17,3916,44100
7,BMW,1 Series,2012,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,High-Performance",Compact,Coupe,28,20,3916,39300
8,BMW,1 Series,2012,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,Luxury,Compact,Convertible,28,18,3916,36900
9,BMW,1 Series,2013,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,Luxury,Compact,Convertible,27,18,3916,37200
